# GPT-2 L8/C396: programming and API specification language

The paper's manual audit labels this component **Programming/API specification language** with confidence 5/5. This quick test compares several programming/API prompts with matched non-programming controls. ICA scores are continuous evidence, not a perfect concept classifier.

In [ ]:
%pip install -q icalens==0.4.0 matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from icalens import ICALens

lens = ICALens.from_pretrained("sida/icalens-gpt2-small-pile10k")
layer, component = 8, 396
profile = lens.component_profile(layer=layer, component=component)
print("Manual label: Programming/API specification language")
print("Selected semantic tail:", profile["tail_direction"])
display(profile)

In [ ]:
prompts = {
    "programming · module spec": 'Require all modules ending in ".spec" and export the createClient interface.',
    "programming · API": "The API accepts a JSON array and returns an error when the token is missing.",
    "programming · schema": "Define the function signature, optional parameters, and the response object.",
    "control · travel": "The train crossed the valley while evening light faded behind the hills.",
    "control · cooking": "Whisk the eggs with milk, then cook them slowly in a warm pan.",
    "control · history": "The treaty was signed after years of negotiation between the two countries.",
}

direction = 1 if profile["tail_direction"] == "positive" else -1
rows = []
analyses = {}
for label, text in prompts.items():
    result = lens.analyze(text, layer=layer, selected_components=[component])
    oriented = direction * result.scores[:, component].detach().float().cpu().numpy()
    rows.append((label, float(np.max(oriented)), float(np.mean(np.sort(oriented)[-min(3,len(oriented)):]))))
    analyses[label] = result

for label, maximum, top3 in rows:
    print(f"{label:28s} max={maximum:+6.2f}  top-3 mean={top3:+6.2f}")

labels = [row[0] for row in rows]; values = [row[2] for row in rows]
colors = ["#057c75" if label.startswith("programming") else "#9aa6b2" for label in labels]
plt.figure(figsize=(9,4)); plt.barh(labels[::-1], values[::-1], color=colors[::-1])
plt.xlabel("Mean of top three C396 scores on the selected semantic tail")
plt.axvline(0,color="#172026",linewidth=.8); plt.tight_layout(); plt.show()

In [ ]:
# Inspect token-aligned evidence for one positive example and one control.
display(analyses["programming · API"])
display(analyses["control · travel"])